In [1]:
#Библиотеки
import csv
import pdb
import random
import requests
import json
import os
#pdb.set_trace()

In [2]:
#Чтение файла
def ReadFile(file, paintings):   
    with open(file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('Classification') == 'Paintings':
                object_number_key = reader.fieldnames[0]
                painting = { 
                    'object_id': row.get('Object ID'),
                    'object_number': row.get(object_number_key), 
                    'title': row.get('Title'),
                    'artist': row.get('Artist Display Name'),
                    'date': row.get('Object Date')
                }
                paintings.append(painting)
    
    print(f"Найдено {len(paintings)} картин")
    
    if not paintings:
        print("Картины не найдены!")
        return
    return paintings

In [3]:
#Выбор случайной картины 
def RandomPainting(paintings):
    random_painting = random.choice(paintings)
    object_id = random_painting['object_id']

    print(f"Номер объекта: {random_painting['object_number']}")
    print(f"Название: {random_painting['title']}")
    print(f"Художник: {random_painting['artist']}")
    print(f"Дата: {random_painting['date']}")
    return object_id

In [4]:
#Получение подробной информации и изображения
def ObjectDetails(object_id):
    url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"
    print(f"\nИнформация: {url}")
    
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Ошибка при запросе: {response.status_code}")
        return
    
    object_data = response.json()
    
    image_url = object_data.get('primaryImage')
    
    if not image_url:
        print("Изображение не найдено")
    else:
        print(f"Найдено изображение: {image_url}")
    return object_data

In [5]:
#Скачивание изображения и информации о нём
def DonlowdData(object_data):
    paintings_dir = 'paintings'
    os.makedirs(paintings_dir, exist_ok=True)
    
    #Формирование имён файлов
    object_id = object_data.get('objectID')

    image_filename = os.path.join(paintings_dir, f"{object_id}.jpg")
    json_filename = os.path.join(paintings_dir, f"{object_id}.json")
    
    #Скачивание изображения
    image_url = object_data.get('primaryImage')
    if image_url or image_url != '':
        img_response = requests.get(image_url, stream=True)

    if image_url != '' and img_response.status_code == 200:
        with open(image_filename, 'wb') as f:
            for chunk in img_response.iter_content(chunk_size=1024):
                f.write(chunk)
        print(f"Изображение сохранено: {image_filename}")
    
    #Сохранение данных
    with open(json_filename, 'w', encoding='utf-8') as f:
        data = {
            'object_id': object_data.get('Object ID'),
            'title': object_data.get('title'),
            'artist': object_data.get('artistDisplayName'),
            'date': object_data.get('objectDate'),
            'medium': object_data.get('medium'),
            'dimensions': object_data.get('dimensions'),
            'classification': object_data.get('classification'),
            'department': object_data.get('department'),
            'image_url': image_url
        }
        json.dump(data, f, indent=2, ensure_ascii=False)

In [6]:
#Main
if __name__ == "__main__":
    file = "MetObjects.csv"
    paintings = []
    #Чтение файла
    paintings = ReadFile(file, paintings)
    #Рандомный номер
    object_id = RandomPainting(paintings)
    #Получение информации
    object_data = ObjectDetails(object_id)
    #Скачивание
    if object_data:
        print("Данные успешно получены")
        DonlowdData(object_data)
    else:
        print("Не удалось получить данные по API")

Найдено 9005 картин
Номер объекта: 1975.1.87
Название: The Molo, Venice, from the Bacino di San Marco
Художник: Luca Carlevaris
Дата: ca. 1709

Информация: https://collectionapi.metmuseum.org/public/collection/v1/objects/459029
Найдено изображение: https://images.metmuseum.org/CRDImages/rl/original/DT3061.jpg
Данные успешно получены
Изображение сохранено: paintings\459029.jpg
